# Notebook 05 — Semantic Embeddings + Qdrant Similarity Search

**Text field:** Movie `overview` (plot description)  
**Embedding model:** `all-MiniLM-L6-v2` (sentence-transformers, runs on CPU, no API key)  
**Vector store:** Qdrant (local Docker)

**Use case:** A studio exec enters a short concept description (e.g., *"space opera with political intrigue"*) and instantly retrieves the 10 most semantically similar movies from the TMDB catalogue — enabling rapid competitive analysis and greenlight decisions.

> Pre-requisite: `docker-compose up -d` running. Qdrant at `http://localhost:6333`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.http.models import (
    Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue
)
import uuid

DATA_DIR = Path("../data")

# Load model (downloads on first run, ~80 MB)
MODEL_NAME = "all-MiniLM-L6-v2"
print(f"Loading embedding model: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)
print(f"Model loaded. Vector dimension: {model.get_sentence_embedding_dimension()}")

/Users/nidhichaubey/Downloads/tmdb_capstone 2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model: all-MiniLM-L6-v2
Model loaded. Vector dimension: 384


## Step 1 — Load Clean Data

In [2]:
df = pd.read_parquet(DATA_DIR / "clean_movies.parquet")

# Filter to movies with meaningful overviews
df_embed = df[
    df['overview'].notna() &
    (df['overview'].str.strip() != '') &
    (df['overview'].str.strip() != 'unknown') &
    (df['overview'].str.len() > 30)
].copy()

# Build enriched text field: title + tagline + overview
df_embed['embed_text'] = (
    df_embed['title'].fillna('') + '. ' +
    df_embed['tagline'].fillna('') + ' ' +
    df_embed['overview'].fillna('')
).str.strip()

print(f"Movies to embed: {len(df_embed):,}")
df_embed[['title', 'embed_text']].head(3)

Movies to embed: 202,164


,title,embed_text
0,Life in Loops (A Megacities RMX),Life in Loops (A Megacities RMX). A Megacities...
1,Finding Nemo,Finding Nemo. There are 3.7 trillion fish in t...
2,Dancer in the Dark,"Dancer in the Dark. In a world of shadows, she..."


## Step 2 — Generate Embeddings

In [3]:
texts = df_embed['embed_text'].tolist()

print(f"Encoding {len(texts):,} movies...")
embeddings = model.encode(
    texts,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"Embeddings shape: {embeddings.shape}")

Encoding 202,164 movies...


Batches: 100%|██████████| 790/790 [26:22<00:00,  2.00s/it]  


Embeddings shape: (202164, 384)


## Step 3 — Store in Qdrant

In [4]:
client = QdrantClient(host="localhost", port=6333)

COLLECTION = "tmdb_movies"
DIM        = embeddings.shape[1]

# Recreate collection (idempotent)
if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)
    print(f"Deleted existing collection: {COLLECTION}")

client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=DIM, distance=Distance.COSINE)
)
print(f"Collection '{COLLECTION}' created. Dim={DIM}")

Deleted existing collection: tmdb_movies
Collection 'tmdb_movies' created. Dim=384


In [5]:
BATCH_SIZE = 500
records = df_embed.reset_index(drop=True)

for i in range(0, len(records), BATCH_SIZE):
    batch = records.iloc[i:i+BATCH_SIZE]
    vecs  = embeddings[i:i+BATCH_SIZE]

    points = [
        PointStruct(
            id=int(row['movie_id']) if pd.notna(row.get('movie_id')) else i + j,
            vector=vecs[j].tolist(),
            payload={
                "movie_id":      int(row['movie_id']) if pd.notna(row.get('movie_id')) else 0,
                "title":         str(row.get('title', '')),
                "overview":      str(row.get('overview', ''))[:500],
                "release_year":  int(row['release_year']) if pd.notna(row.get('release_year')) else 0,
                "vote_average":  float(row['vote_average']) if pd.notna(row.get('vote_average')) else None,
                "genres":        str(row.get('genres', '')),
                "director":      str(row.get('director', '')),
                "cast_list":     str(row.get('cast_list', '')),
                "is_successful": int(row.get('is_successful', 0)),
                "revenue":       float(row['revenue']) if pd.notna(row.get('revenue')) and row.get('revenue', 0) > 0 else None,
                "budget":        float(row['budget'])  if pd.notna(row.get('budget'))  and row.get('budget', 0)  > 0 else None,
                "roi_pct":       float(row['roi_pct']) if pd.notna(row.get('roi_pct')) else None,
            }
        )
        for j, (_, row) in enumerate(batch.iterrows())
    ]

    client.upsert(collection_name=COLLECTION, points=points)
    if i % 5000 == 0:
        print(f"  Uploaded {i + len(batch):,} / {len(records):,}")

info = client.get_collection(COLLECTION)
print(f"\n✅ Collection '{COLLECTION}': {info.points_count:,} vectors indexed")
print("Payload fields stored: movie_id, title, overview, release_year, vote_average,")
print("  genres, director, cast_list, is_successful, revenue, budget, roi_pct")


  Uploaded 500 / 202,164
  Uploaded 5,500 / 202,164
  Uploaded 10,500 / 202,164
  Uploaded 15,500 / 202,164
  Uploaded 20,500 / 202,164
  Uploaded 25,500 / 202,164
  Uploaded 30,500 / 202,164
  Uploaded 35,500 / 202,164
  Uploaded 40,500 / 202,164
  Uploaded 45,500 / 202,164
  Uploaded 50,500 / 202,164
  Uploaded 55,500 / 202,164
  Uploaded 60,500 / 202,164
  Uploaded 65,500 / 202,164
  Uploaded 70,500 / 202,164
  Uploaded 75,500 / 202,164
  Uploaded 80,500 / 202,164
  Uploaded 85,500 / 202,164
  Uploaded 90,500 / 202,164
  Uploaded 95,500 / 202,164
  Uploaded 100,500 / 202,164
  Uploaded 105,500 / 202,164
  Uploaded 110,500 / 202,164
  Uploaded 115,500 / 202,164
  Uploaded 120,500 / 202,164
  Uploaded 125,500 / 202,164
  Uploaded 130,500 / 202,164
  Uploaded 135,500 / 202,164
  Uploaded 140,500 / 202,164
  Uploaded 145,500 / 202,164
  Uploaded 150,500 / 202,164
  Uploaded 155,500 / 202,164
  Uploaded 160,500 / 202,164
  Uploaded 165,500 / 202,164
  Uploaded 170,500 / 202,164
  Uploade

## Step 4 — Similarity Search Demo

In [6]:
def search_movies(query_text, top_k=10, min_rating=0.0):
    """Embed a query and return the top-k most similar movies."""
    query_vec = model.encode([query_text], convert_to_numpy=True)[0]
    
    results = client.search(
        collection_name=COLLECTION,
        query_vector=query_vec.tolist(),
        limit=top_k,
        with_payload=True
    )
    
    rows = []
    for r in results:
        p = r.payload
        if p.get('vote_average', 0) >= min_rating:
            rows.append({
                'title':        p.get('title'),
                'score':        round(r.score, 4),
                'year':         p.get('release_year'),
                'rating':       p.get('vote_average'),
                'genres':       p.get('genres', '')[:60],
                'director':     p.get('director', ''),
                'overview':     p.get('overview', '')[:120] + '...'
            })
    return pd.DataFrame(rows)

In [7]:
# Test Query 1 — Space Opera with political intrigue
df_r = search_movies("space opera with political intrigue and epic battles", top_k=10)
print("=== Query: 'space opera with political intrigue' ===")
print(df_r[['title', 'score', 'year', 'rating', 'genres']].to_string(index=False))

=== Query: 'space opera with political intrigue' ===
                                                    title  score  year  rating                                        genres
                                John Vardar vs the Galaxy 0.6400  2024     7.0 Animation, Adventure, Comedy, Science Fiction
                          Siegfried - San Francisco Opera 0.5949  2018     0.0                                         Music
Astronome: A Night at the Opera (A Disturbing Initiation) 0.5723  2009     0.0                                         Music
                                   Shoestring Space Opera 0.5618  2011     0.0                                   Documentary
                                    Opera Australia: Aida 0.5527  2015     0.0                History, Music, Drama, Romance
                           The Space: Theatre of Survival 0.5498  2019     0.0                                   Documentary
                                 Prokofiev: War and Peace 0.5380  2023  

In [8]:
# Test Query 2 — Psychological thriller
df_r2 = search_movies("psychological thriller about identity and memory loss", top_k=10)
print("=== Query: 'psychological thriller identity memory' ===")
print(df_r2[['title', 'score', 'year', 'rating', 'genres']].to_string(index=False))

=== Query: 'psychological thriller identity memory' ===
               title  score  year  rating                                    genres
            Amnesiac 0.6659  2015     4.6          Thriller, Mystery, Drama, Horror
Memoir of a Murderer 0.6604  2017     7.4                  Crime, Mystery, Thriller
        Amnesia Love 0.6385  2018     7.0                                    Comedy
  Battle of Memories 0.6385  2017     6.6 Drama, Mystery, Thriller, Science Fiction
            Memories 0.6367  2002     7.0                           Horror, Mystery
     Amnesia Diaries 0.6060  2012     0.0                      History, Documentary
      Memory of Fear 0.5994  2017     0.0                                      None
         Blank Slate 0.5980  2008     6.9          Crime, Thriller, Drama, TV Movie
      Forgotten Evil 0.5946  2017     4.8                        Thriller, TV Movie
     Faces of Deceit 0.5932  2018     0.0                           Thriller, Drama


In [9]:
# Test Query 3 — Heartwarming family adventure
df_r3 = search_movies("heartwarming animated family adventure with talking animals", top_k=10)
print("=== Query: 'animated family adventure talking animals' ===")
print(df_r3[['title', 'score', 'year', 'rating', 'genres']].to_string(index=False))

print("\n✅ Qdrant similarity search working.")

=== Query: 'animated family adventure talking animals' ===
                                                                    title  score  year  rating                      genres
                                                         Meet the Pegasus 0.5833  2014     6.7   Family, Animation, Comedy
                                              The Barkers: Mind the Cats! 0.5688  2020     6.0                   Animation
                                Animal Kingdom 3D: A Tale of Six Families 0.5604  2023     0.0                 Documentary
                                                 PBS Kids: 20 Furry Tales 0.5588  2015    10.0 Animation, Family, TV Movie
                                                                    Earth 0.5520  2007     7.6                 Documentary
                                                   Jack and the Beanstalk 0.5396  2020     0.0                      Family
Walt Disney Animation Collection: Classic Short Films - Three Little Pigs 0.5386